# Train CNN Model on Local Machine
Huấn luyện mô hình CNN với PyTorch trên máy cá nhân

**Yêu cầu**: Đã tải dữ liệu từ server vào thư mục `data_for_training/`

In [ ]:
import os
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import geopandas as gpd
import joblib
from utils import load_data_geo

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

## Load dữ liệu từ server

In [ ]:
# Đường dẫn dữ liệu
data_dir = "data_for_training"

print("📂 Kiểm tra các file dữ liệu...")
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"❌ Thư mục '{data_dir}' không tồn tại. Hãy tải dữ liệu từ server trước.")

# Load dữ liệu
print("\n📥 Load dữ liệu...")
average_ndvi = xr.open_dataarray(os.path.join(data_dir, "average_ndvi.nc"))
average_vv = xr.open_dataarray(os.path.join(data_dir, "average_vv.nc"))
average_vh = xr.open_dataarray(os.path.join(data_dir, "average_vh.nc"))

print(f"✅ NDVI shape: {average_ndvi.shape}")
print(f"✅ VV shape: {average_vv.shape}")
print(f"✅ VH shape: {average_vh.shape}")

In [ ]:
# Load training data
train_data_path = os.path.join(data_dir, "train_data", "ST_training data_updated_1130points_new.shp")
print(f"\n📋 Load training points...")
train = load_data_geo(train_data_path)
print(f"✅ Training points: {len(train)}")
print(f"✅ Columns: {train.columns.tolist()}")
train.head()

## Chuẩn bị dữ liệu huấn luyện

In [ ]:
# Cấu hình nhãn dữ liệu
label_mapping = {
    "Lua tom": 0,
    "Lua": 1,
    "CHN": 2,
    "CLN": 3,
    "TS": 4,
    "Song": 5,
    "Dat xay dung": 6,
    "Rung": 7,
}

num_classes = len(label_mapping)
print(f"🏷️  Số lớp: {num_classes}")
for label, idx in label_mapping.items():
    print(f"   {idx}: {label}")

In [ ]:
# Trích xuất dữ liệu từ các điểm training
print("\n🔧 Trích xuất dữ liệu từ các điểm...")
X = []
y = []

for idx, point in train.iterrows():
    try:
        # Lấy tọa độ
        x_coord = point.geometry.x
        y_coord = point.geometry.y
        
        # Trích xuất giá trị từ mỗi band
        ndvi_data = average_ndvi.sel(x=x_coord, y=y_coord, method='nearest').values
        vv_data = average_vv.sel(x=x_coord, y=y_coord, method='nearest').values
        vh_data = average_vh.sel(x=x_coord, y=y_coord, method='nearest').values
        
        # Concatenate các band
        data_point = np.concatenate((ndvi_data, vv_data, vh_data))
        X.append(data_point)
        
        # Lấy nhãn
        label_text = point.Hientrang
        label_idx = label_mapping[label_text]
        y.append(label_idx)
        
    except Exception as e:
        print(f"   ⚠️  Point {idx}: {e}")

X = np.array(X)
y = np.array(y)

print(f"✅ Dữ liệu trích xuất: {X.shape}")
print(f"✅ Nhãn: {y.shape}")
print(f"✅ Phân phối nhãn: {np.bincount(y)}")

In [ ]:
# Chia dữ liệu
print("\n📊 Chia dữ liệu (train/val/test = 60/20/20)...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"✅ Train: {X_train.shape}")
print(f"✅ Val: {X_val.shape}")
print(f"✅ Test: {X_test.shape}")

In [ ]:
# Normalize dữ liệu
print("\n🔧 Normalize dữ liệu...")
mean = X_train.mean()
std = X_train.std()
X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

print(f"✅ Mean: {mean:.4f}")
print(f"✅ Std: {std:.4f}")

In [ ]:
# Convert to PyTorch tensors
print("\n🔧 Convert sang PyTorch tensors...")
X_train_tensor = torch.FloatTensor(X_train).unsqueeze(1)  # (N, 1, features)
X_val_tensor = torch.FloatTensor(X_val).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1)

y_train_tensor = torch.LongTensor(y_train)
y_val_tensor = torch.LongTensor(y_val)
y_test_tensor = torch.LongTensor(y_test)

print(f"✅ X_train shape: {X_train_tensor.shape}")
print(f"✅ y_train shape: {y_train_tensor.shape}")

In [ ]:
# Tạo DataLoaders
print("\n🔧 Tạo DataLoaders...")
batch_size = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Val batches: {len(val_loader)}")
print(f"✅ Test batches: {len(test_loader)}")

## Xây dựng CNN Model

In [ ]:
# Định nghĩa CNN model
class CNNClassifier(nn.Module):
    def __init__(self, input_size, num_classes=8):
        super(CNNClassifier, self).__init__()
        
        # Conv blocks
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2)
        self.drop1 = nn.Dropout(0.25)
        
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.conv4 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2)
        self.drop2 = nn.Dropout(0.25)
        
        self.conv5 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm1d(256)
        self.conv6 = nn.Conv1d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm1d(256)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.drop3 = nn.Dropout(0.25)
        
        # FC layers
        self.fc1 = nn.Linear(256, 128)
        self.bn_fc1 = nn.BatchNorm1d(128)
        self.drop_fc1 = nn.Dropout(0.5)
        
        self.fc2 = nn.Linear(128, 64)
        self.bn_fc2 = nn.BatchNorm1d(64)
        self.drop_fc2 = nn.Dropout(0.5)
        
        self.fc3 = nn.Linear(64, num_classes)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.drop1(x)
        
        # Block 2
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.drop2(x)
        
        # Block 3
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.relu(self.bn6(self.conv6(x)))
        x = self.global_avg_pool(x)
        x = x.squeeze(-1)
        x = self.drop3(x)
        
        # FC layers
        x = self.relu(self.bn_fc1(self.fc1(x)))
        x = self.drop_fc1(x)
        x = self.relu(self.bn_fc2(self.fc2(x)))
        x = self.drop_fc2(x)
        x = self.fc3(x)
        
        return x

# Khởi tạo model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNClassifier(input_size=X_train.shape[1], num_classes=num_classes)
model = model.to(device)

print(f"✅ Model created")
print(f"📍 Device: {device}")
print(f"\n📋 Model architecture:")
print(model)

## Huấn luyện Model

In [ ]:
# Loss function và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

epochs = 100
early_stop_patience = 15
best_val_loss = float('inf')
early_stop_counter = 0

train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("🚀 Bắt đầu huấn luyện...\n")

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == y_batch).sum().item()
        train_total += y_batch.size(0)
    
    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_correct += (predicted == y_batch).sum().item()
            val_total += y_batch.size(0)
    
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%')
    
    # Early stopping
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        # Lưu best model
        torch.save(model.state_dict(), 'model_cnn_pytorch_best.pt')
    else:
        early_stop_counter += 1
        if early_stop_counter >= early_stop_patience:
            print(f'\n⛔ Early stopping at epoch {epoch+1}')
            break

print(f'\n✅ Huấn luyện hoàn thành!')

In [ ]:
# Load best model
model.load_state_dict(torch.load('model_cnn_pytorch_best.pt'))

# Evaluate on test set
model.eval()
test_correct = 0
test_total = 0
test_loss = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        test_correct += (predicted == y_batch).sum().item()
        test_total += y_batch.size(0)

test_loss /= len(test_loader)
test_acc = 100 * test_correct / test_total

print(f"📊 Test Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

In [ ]:
# Vẽ đồ thị huấn luyện
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(train_accs, label='Train Accuracy')
axes[1].plot(val_accs, label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Model Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Đồ thị đã lưu vào 'training_history.png'")

## Lưu Model

In [ ]:
# Lưu model state dict
print("💾 Lưu model...")
torch.save(model.state_dict(), 'model_cnn_pytorch.pt')
print(f"✅ Model state dict đã lưu: model_cnn_pytorch.pt")

# Lưu thông tin model
model_info = {
    'state_dict': model.state_dict(),
    'num_classes': num_classes,
    'input_size': X_train.shape[1],
    'label_mapping': label_mapping,
    'mean': mean,
    'std': std,
    'test_accuracy': test_acc,
    'test_loss': test_loss
}

torch.save(model_info, 'model_cnn_pytorch_full.pt')
print(f"✅ Full model info đã lưu: model_cnn_pytorch_full.pt")

print(f"\n✅ Model ready for prediction!")